In [8]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler
from hmmlearn.hmm import GaussianHMM


In [9]:
# ==========================================================
# 1) LOAD DATA
# ==========================================================
prices_m2 = pd.read_csv("../Data/prices_round_0_day_-2.csv", sep=";")
prices_m1 = pd.read_csv("../Data/prices_round_0_day_-1.csv", sep=";")

df_prices = pd.concat([prices_m2, prices_m1], ignore_index=True)
df_prices = df_prices.sort_values(["day", "timestamp", "product"]).reset_index(drop=True)


In [10]:
# ==========================================================
# 2) FEATURE ENGINEERING
# ==========================================================
def build_hmm_features(df_prices: pd.DataFrame, product_name: str, h: int = 1) -> pd.DataFrame:
    df = (
        df_prices[df_prices["product"] == product_name]
        .copy()
        .sort_values(["day", "timestamp"])
        .reset_index(drop=True)
    )

    df["mid"] = (df["bid_price_1"] + df["ask_price_1"]) / 2

    df["ask_volume_1_abs"] = df["ask_volume_1"].abs()
    df["ask_volume_2_abs"] = df["ask_volume_2"].abs() if "ask_volume_2" in df.columns else 0
    df["ask_volume_3_abs"] = df["ask_volume_3"].abs() if "ask_volume_3" in df.columns else 0

    den1 = df["bid_volume_1"] + df["ask_volume_1_abs"]
    df["microprice"] = np.where(
        den1 > 0,
        (df["bid_price_1"] * df["ask_volume_1_abs"] + df["ask_price_1"] * df["bid_volume_1"]) / den1,
        df["mid"]
    )

    df["spread"] = df["ask_price_1"] - df["bid_price_1"]

    df["imbalance_1"] = np.where(
        den1 > 0,
        (df["bid_volume_1"] - df["ask_volume_1_abs"]) / den1,
        0.0
    )

    bid_v2 = df["bid_volume_2"].fillna(0) if "bid_volume_2" in df.columns else 0
    ask_v2 = df["ask_volume_2_abs"].fillna(0) if "ask_volume_2_abs" in df.columns else 0
    den2 = bid_v2 + ask_v2
    df["imbalance_2"] = np.where(
        den2 > 0,
        (bid_v2 - ask_v2) / den2,
        0.0
    )

    bid_v3 = df["bid_volume_3"].fillna(0) if "bid_volume_3" in df.columns else 0
    ask_v3 = df["ask_volume_3_abs"].fillna(0) if "ask_volume_3_abs" in df.columns else 0

    df["bid_depth_tot"] = df["bid_volume_1"].fillna(0) + bid_v2 + bid_v3
    df["ask_depth_tot"] = df["ask_volume_1_abs"].fillna(0) + ask_v2 + ask_v3

    den_tot = df["bid_depth_tot"] + df["ask_depth_tot"]
    df["imbalance_tot"] = np.where(
        den_tot > 0,
        (df["bid_depth_tot"] - df["ask_depth_tot"]) / den_tot,
        0.0
    )

    df["micro_edge"] = df["microprice"] - df["mid"]

    g = df.groupby("day", group_keys=False)

    df["ret_1"] = g["mid"].diff()
    df["mid_fut"] = g["mid"].shift(-h)
    df["ret_fut"] = df["mid_fut"] - df["mid"]

    return df

In [11]:
# ==========================================================
# 3) FIT HMM
# ==========================================================
def fit_hmm_states(df_feat: pd.DataFrame, feature_cols: list[str], n_states: int = 3):
    model_df = df_feat[["day", "timestamp", "mid", "mid_fut", "ret_fut"] + feature_cols].dropna().copy()

    train = model_df[model_df["day"] == -2].copy()
    test = model_df[model_df["day"] == -1].copy()

    scaler = RobustScaler()

    X_train = scaler.fit_transform(train[feature_cols])
    X_test = scaler.transform(test[feature_cols])

    hmm = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=200,
        random_state=42
    )
    hmm.fit(X_train)

    train_states = hmm.predict(X_train)
    test_states = hmm.predict(X_test)

    train["state_raw"] = train_states
    test["state_raw"] = test_states

    return hmm, scaler, train, test


In [12]:
# ==========================================================
# 4) LABEL STATES AS BULLISH / NEUTRAL / BEARISH
# ==========================================================
def label_states_by_future_return(train_df: pd.DataFrame):
    state_summary = (
        train_df.groupby("state_raw")
        .agg(
            count=("state_raw", "size"),
            avg_ret_fut=("ret_fut", "mean"),
            avg_ret_1=("ret_1", "mean"),
            avg_micro_edge=("micro_edge", "mean"),
            avg_imb1=("imbalance_1", "mean"),
            avg_imbtot=("imbalance_tot", "mean"),
            avg_spread=("spread", "mean"),
        )
        .reset_index()
        .sort_values("avg_ret_fut")
        .reset_index(drop=True)
    )

    if len(state_summary) != 3:
        raise ValueError("Expected exactly 3 states")

    bearish_state = int(state_summary.iloc[0]["state_raw"])
    neutral_state = int(state_summary.iloc[1]["state_raw"])
    bullish_state = int(state_summary.iloc[2]["state_raw"])

    label_map = {
        bearish_state: "bearish",
        neutral_state: "neutral",
        bullish_state: "bullish",
    }

    state_summary["label"] = state_summary["state_raw"].map(label_map)

    return label_map, state_summary


In [13]:
# ==========================================================
# 5) APPLY LABELS AND SUMMARIZE TEST SET
# ==========================================================
def summarize_test_states(test_df: pd.DataFrame, label_map: dict):
    test_df = test_df.copy()
    test_df["state_label"] = test_df["state_raw"].map(label_map)

    summary = (
        test_df.groupby("state_label")
        .agg(
            count=("state_label", "size"),
            avg_ret_fut=("ret_fut", "mean"),
            avg_ret_1=("ret_1", "mean"),
            avg_micro_edge=("micro_edge", "mean"),
            avg_imb1=("imbalance_1", "mean"),
            avg_imbtot=("imbalance_tot", "mean"),
            avg_spread=("spread", "mean"),
        )
        .reset_index()
    )

    return test_df, summary


In [14]:
# ==========================================================
# 6) RUN
# ==========================================================
feature_cols = [
    "ret_1",
    "micro_edge",
    "imbalance_1",
    "imbalance_tot",
    "spread",
]

df_feat = build_hmm_features(df_prices, "TOMATOES", h=1)

hmm, scaler, train_states_df, test_states_df = fit_hmm_states(
    df_feat=df_feat,
    feature_cols=feature_cols,
    n_states=3,
)

label_map, train_state_summary = label_states_by_future_return(train_states_df)
test_labeled_df, test_state_summary = summarize_test_states(test_states_df, label_map)

print("\n" + "=" * 80)
print("TRAIN STATE SUMMARY")
print("=" * 80)
print(train_state_summary)

print("\n" + "=" * 80)
print("TEST STATE SUMMARY")
print("=" * 80)
print(test_state_summary)

print("\n" + "=" * 80)
print("STATE LABEL MAP")
print("=" * 80)
print(label_map)


TRAIN STATE SUMMARY
   state_raw  count  avg_ret_fut  avg_ret_1  avg_micro_edge  avg_imb1  \
0          0    359    -2.959610   2.949861       -0.810127 -0.190129   
1          1   9277    -0.016439   0.019996        0.000000  0.000000   
2          2    362     3.367403  -3.418508        0.716367  0.191721   

   avg_imbtot  avg_spread    label  
0    0.091016    7.676880  bearish  
1    0.000000   13.527002  neutral  
2   -0.090998    6.571823  bullish  

TEST STATE SUMMARY
  state_label  count  avg_ret_fut  avg_ret_1  avg_micro_edge  avg_imb1  \
0     bearish    364    -2.880495   2.920330       -0.816609 -0.196756   
1     bullish    359     3.231198  -3.295265        0.670856  0.179799   
2     neutral   9275    -0.017358   0.007709        0.000000  0.000000   

   avg_imbtot  avg_spread  
0    0.088771    7.500000  
1   -0.092482    6.568245  
2    0.000000   13.437951  

STATE LABEL MAP
{0: 'bearish', 1: 'neutral', 2: 'bullish'}
